# 📊 EDA — Hệ thống Dự báo Chất lượng Không khí Hà Nội 2022–2025
> **Nhóm:** Nguyễn Thị Phương Anh · Bùi Quang Hùng · Lưu Linh Linh · Tạ Minh Trường  
> **Dữ liệu:** 30,000+ bản ghi theo giờ, Hà Nội, 2022 – đầu 2025  
> **Mục tiêu EDA:** Khám phá phân phối AQI theo thời gian, mùa, giờ cao điểm và tương quan các feature

## 0. Import thư viện & cấu hình

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from numpy.polynomial.polynomial import polyfit
import warnings
warnings.filterwarnings('ignore')
from sqlalchemy import create_engine

sns.set_theme(style='whitegrid', font='DejaVu Sans')
plt.rcParams.update({'figure.dpi': 130, 'axes.titlepad': 10})

# ── Màu sắc AQI category ──────────────────────────────────────────────────
PALETTE = {
    'Good':                           '#2ecc71',
    'Moderate':                       '#f1c40f',
    'Unhealthy for sensitive groups': '#e67e22',
    'Unhealthy':                      '#e74c3c',
    'Very Unhealthy':                 '#9b59b6',
    'Hazardous':                      '#7f1d1d',
}
CAT_ORDER = list(PALETTE.keys())

# ── Đồng bộ với README: 0=Đông, 1=Xuân, 2=Hạ, 3=Thu ─────────────────────
SEASON_MAP   = {0: 'Đông', 1: 'Xuân', 2: 'Hạ', 3: 'Thu'}
SEASON_ORDER = ['Xuân', 'Hạ', 'Thu', 'Đông']
SEASON_CLR   = {'Xuân': '#27ae60', 'Hạ': '#e74c3c',
                'Thu':  '#e67e22', 'Đông': '#3498db'}

print("✅ Import xong!")

## 1. Kết nối MySQL & đọc dữ liệu

In [ ]:
from sqlalchemy import create_engine

# ✏️ Chỉnh host/password nếu cần
DB_USER     = 'root'
DB_PASSWORD = 'minhtruong123.'
DB_HOST     = 'localhost'
DB_PORT     = 3306
DB_NAME     = 'hanoi_aqi'
TABLE_NAME  = 'aqi_data'

engine = create_engine(f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

print(f"Đang kết nối MySQL: {DB_HOST}:{DB_PORT}/{DB_NAME} ...")
df = pd.read_sql(f'SELECT * FROM {TABLE_NAME}', con=engine)
print(f"✅ Đọc xong: {df.shape[0]:,} hàng × {df.shape[1]} cột")

df.columns = df.columns.str.strip().str.lower()
df['aqi_category'] = pd.Categorical(df['aqi_category'], categories=CAT_ORDER, ordered=True)
df['season_name']  = df['season'].map(SEASON_MAP)
df['ym'] = pd.to_datetime(df['year'].astype(str) + '-' + df['month'].astype(str).str.zfill(2))

print(f"Thời gian: {df['local_time'].min()} → {df['local_time'].max()}")
print()
print("Phân bố AQI Category:")
counts = df['aqi_category'].value_counts()
for cat, v in counts.items():
    print(f"  {cat:<38}: {v:>6,}  ({v/len(df)*100:.1f}%)")
df.head(3)

## 2. EDA 1 — Phân phối AQI trung bình theo giờ trong ngày
Khám phá giờ nào trong ngày Hà Nội ô nhiễm nhất / sạch nhất.

In [ ]:
hourly = df.groupby('hour')['aqi'].mean()
peak_h = int(hourly.idxmax())
best_h = int(hourly.idxmin())

fig, ax = plt.subplots(figsize=(13, 4.5))

# Tô nền giờ cao điểm: 6-9h và 17-20h
for start, end in [(6, 9), (17, 20)]:
    ax.axvspan(start, end, alpha=0.10, color='#e74c3c', label='Giờ cao điểm' if start==6 else '')

ax.plot(hourly.index, hourly.values, color='#e74c3c', lw=2.5, marker='o', ms=5, zorder=3)
ax.fill_between(hourly.index, hourly.values, alpha=0.12, color='#e74c3c')
ax.set_xticks(range(24))
ax.set_xlabel('Giờ trong ngày', fontsize=12)
ax.set_ylabel('AQI trung bình', fontsize=12)
ax.set_title('Phân phối AQI trung bình theo giờ trong ngày — Hà Nội 2022–2025',
             fontsize=13, fontweight='bold')

ax.annotate(f'Ô nhiễm nhất\n{peak_h}h  (AQI={hourly[peak_h]:.1f})',
            xy=(peak_h, hourly[peak_h]), xytext=(peak_h+1.5, hourly[peak_h]+4),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', fc='#ffe6e6', ec='#e74c3c'))
ax.annotate(f'Sạch nhất\n{best_h}h  (AQI={hourly[best_h]:.1f})',
            xy=(best_h, hourly[best_h]), xytext=(best_h+1.5, hourly[best_h]-8),
            arrowprops=dict(arrowstyle='->', color='green'), fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', fc='#e8f8f5', ec='#27ae60'))

handles = [plt.Rectangle((0,0),1,1, fc='#e74c3c', alpha=0.2)]
ax.legend(handles, ['Khung giờ cao điểm (6–9h & 17–20h)'], fontsize=9, loc='upper left')
plt.tight_layout()
plt.show()

print(f"→ Giờ ô nhiễm nhất: {peak_h}h  (AQI={hourly[peak_h]:.1f})")
print(f"→ Giờ sạch nhất:    {best_h}h  (AQI={hourly[best_h]:.1f})")

## 3. EDA 2 — Boxplot AQI theo 4 mùa
Đồng bộ theo README: **0=Đông · 1=Xuân · 2=Hạ · 3=Thu**

In [ ]:
season_stats = df.groupby('season_name')['aqi'].agg(['mean', 'median', 'std'])
worst_s = season_stats['mean'].idxmax()
best_s  = season_stats['mean'].idxmin()

fig, ax = plt.subplots(figsize=(9, 6))
sns.boxplot(data=df, x='season_name', y='aqi', order=SEASON_ORDER,
            palette=SEASON_CLR, width=0.5, linewidth=1.5,
            flierprops=dict(marker='o', markersize=2, alpha=0.3), ax=ax)

ax.set_xlabel('Mùa', fontsize=12)
ax.set_ylabel('AQI', fontsize=12)
ax.set_title('Phân phối AQI theo 4 mùa — Hà Nội 2022–2025\n(0=Đông · 1=Xuân · 2=Hạ · 3=Thu theo README)',
             fontsize=12, fontweight='bold')

# Annotation mean từng mùa
for i, s in enumerate(SEASON_ORDER):
    m = season_stats.loc[s, 'mean']
    ax.text(i, ax.get_ylim()[0] + 3, f'avg\n{m:.0f}', ha='center', fontsize=9, color='#333')

# Highlight mùa tệ nhất
idx = SEASON_ORDER.index(worst_s)
med = season_stats.loc[worst_s, 'median']
ax.annotate(f'AQI cao nhất:\n{worst_s} (median={med:.0f})',
            xy=(idx, med), xytext=(idx+0.45, med+18),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', fc='#fdecea', ec='#e74c3c', alpha=0.9))
plt.tight_layout()
plt.show()

print("→ AQI trung bình theo mùa:")
for s in SEASON_ORDER:
    flag = " ← CAO NHẤT" if s == worst_s else (" ← thấp nhất" if s == best_s else "")
    print(f"   {s}: mean={season_stats.loc[s,'mean']:.1f}  median={season_stats.loc[s,'median']:.1f}  std={season_stats.loc[s,'std']:.1f}{flag}")

## 4. EDA 3 — Heatmap tương quan 13 features

In [ ]:
FEAT13 = ['aqi','co','no2','o3','pm10','pm25','so2',
          'clouds','precipitation','pressure','relative_humidity',
          'temperature','wind_speed']
corr = df[FEAT13].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, linewidths=0.5, annot_kws={'size': 8},
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Heatmap tương quan 13 features — Hà Nội AQI', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

aqi_corr = corr['aqi'].drop('aqi').sort_values(key=abs, ascending=False)
print("→ Top 5 features tương quan mạnh nhất với AQI:")
for feat, val in aqi_corr.head(5).items():
    bar = '█' * int(abs(val)*20)
    print(f"   {feat:<22}: {val:+.3f}  {bar}")

## 5. EDA 4 — Trend AQI trung bình theo tháng 2022–2025

In [ ]:
monthly = df.groupby('ym')['aqi'].mean().reset_index()
x_num   = np.arange(len(monthly))
c_poly  = polyfit(x_num, monthly['aqi'].values, 1)
trend   = c_poly[0] + c_poly[1]*x_num
direction = '↑ TĂNG' if c_poly[1] > 0 else '↓ GIẢM'

yearly = df.groupby('year')['aqi'].mean()

fig, ax = plt.subplots(figsize=(14, 5))
colors_yr = {2022:'#eaf4fb', 2023:'#fef9e7', 2024:'#eafaf1', 2025:'#fdecea'}
for yr, col in colors_yr.items():
    ax.axvspan(pd.Timestamp(f'{yr}-01-01'), pd.Timestamp(f'{yr}-12-31'),
               alpha=0.35, color=col, lw=0)
    ax.text(pd.Timestamp(f'{yr}-07-01'), monthly['aqi'].min()-6,
            str(yr), ha='center', fontsize=9, color='#aaa')

ax.plot(monthly['ym'], monthly['aqi'], color='#2980b9', lw=2,
        marker='o', ms=4, zorder=3, label='AQI trung bình tháng')
ax.fill_between(monthly['ym'], monthly['aqi'], alpha=0.12, color='#2980b9')
ax.plot(monthly['ym'], trend, '--', color='#e74c3c', lw=2,
        label=f'Xu hướng tổng thể: {direction} (slope={c_poly[1]:.2f}/tháng)')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))
plt.xticks(rotation=45, ha='right')
ax.set_xlabel('Tháng', fontsize=12)
ax.set_ylabel('AQI trung bình', fontsize=12)
ax.set_title('Trend AQI trung bình theo tháng — Hà Nội 2022–2025', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"→ Xu hướng: {direction}  (slope={c_poly[1]:.4f}/tháng)")
print("→ AQI trung bình từng năm:")
for yr, v in yearly.items():
    print(f"   {yr}: {v:.1f}")

## 6. EDA 5 — Tỷ lệ 6 mức AQI Category

In [ ]:
cat_counts = df['aqi_category'].value_counts().reindex(CAT_ORDER)
colors = [PALETTE[c] for c in CAT_ORDER]

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

wedges, _, autotexts = axes[0].pie(
    cat_counts, labels=None, colors=colors, autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=1.5), pctdistance=0.78)
for at in autotexts: at.set_fontsize(9)
axes[0].legend(wedges, [f'{c}  ({v:,})' for c, v in zip(CAT_ORDER, cat_counts)],
               loc='lower center', bbox_to_anchor=(0.5,-0.15), fontsize=8, ncol=2)
axes[0].set_title('Tỷ lệ 6 mức AQI Category', fontsize=12, fontweight='bold')

short_labels = ['Good', 'Moderate', 'Unhealthy\nSensitive', 'Unhealthy', 'Very\nUnhealthy', 'Hazardous']
bars = axes[1].bar(range(len(CAT_ORDER)), cat_counts.values, color=colors,
                   edgecolor='white', linewidth=0.8)
axes[1].set_xticks(range(len(CAT_ORDER)))
axes[1].set_xticklabels(short_labels, fontsize=8.5)
axes[1].set_ylabel('Số bản ghi', fontsize=11)
axes[1].set_title('Số lượng theo 6 mức AQI', fontsize=12, fontweight='bold')
for bar, v in zip(bars, cat_counts.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+60,
                 f'{v:,}\n({v/len(df)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=8, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. EDA 6 — AQI giờ cao điểm vs không cao điểm theo từng mùa
Chứng minh ý nghĩa của feature `is_rush_hour` tự tạo: giờ cao điểm có thực sự ô nhiễm hơn không, và mùa nào tác động lớn nhất?

In [ ]:
df['rush_label'] = df['is_rush_hour'].map({1: 'Giờ cao điểm', 0: 'Giờ thường'})

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Subplot 1: Boxplot AQI theo mùa × is_rush_hour ──────────────────────
sns.boxplot(data=df, x='season_name', y='aqi', hue='rush_label',
            order=SEASON_ORDER, palette={'Giờ cao điểm':'#e74c3c','Giờ thường':'#3498db'},
            width=0.6, linewidth=1.2,
            flierprops=dict(marker='o', markersize=1.5, alpha=0.2), ax=axes[0])
axes[0].set_title('AQI: Giờ cao điểm vs Giờ thường theo mùa', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Mùa', fontsize=11)
axes[0].set_ylabel('AQI', fontsize=11)
axes[0].legend(title='Khung giờ', fontsize=9)

# ── Subplot 2: Bar AQI trung bình rush vs normal mỗi mùa ─────────────────
rush_season = df.groupby(['season_name','rush_label'])['aqi'].mean().unstack()
x = np.arange(len(SEASON_ORDER))
w = 0.35
bars1 = axes[1].bar(x-w/2, [rush_season.loc[s,'Giờ cao điểm'] for s in SEASON_ORDER],
                    w, label='Giờ cao điểm', color='#e74c3c', alpha=0.85)
bars2 = axes[1].bar(x+w/2, [rush_season.loc[s,'Giờ thường'] for s in SEASON_ORDER],
                    w, label='Giờ thường', color='#3498db', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(SEASON_ORDER, fontsize=11)
axes[1].set_ylabel('AQI trung bình', fontsize=11)
axes[1].set_title('AQI trung bình: Giờ cao điểm vs Giờ thường theo mùa', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

for bar in list(bars1)+list(bars2):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5,
                 f'{bar.get_height():.0f}', ha='center', fontsize=8.5, fontweight='bold')

plt.tight_layout()
plt.show()

print("→ AQI trung bình (Giờ cao điểm vs Giờ thường) theo mùa:")
print(f"  {'Mùa':<8} {'Cao điểm':>12} {'Thường':>10} {'Chênh lệch':>12}")
print("  " + "-"*45)
for s in SEASON_ORDER:
    rush_v  = rush_season.loc[s,'Giờ cao điểm']
    norm_v  = rush_season.loc[s,'Giờ thường']
    delta   = rush_v - norm_v
    flag = " ↑ cao hơn" if delta > 0 else " ↓ thấp hơn"
    print(f"  {s:<8} {rush_v:>12.1f} {norm_v:>10.1f} {delta:>+11.1f}{flag}")

## 8. EDA 7 — AQI theo thứ trong tuần & cuối tuần
Cuối tuần (ít xe cộ, ít công nghiệp) có AQI thấp hơn ngày thường không?

In [ ]:
DAY_MAP = {0:'Thứ 2', 1:'Thứ 3', 2:'Thứ 4', 3:'Thứ 5', 4:'Thứ 6', 5:'Thứ 7', 6:'CN'}
df['day_name'] = df['day_of_week'].map(DAY_MAP)
DAY_ORDER = ['Thứ 2','Thứ 3','Thứ 4','Thứ 5','Thứ 6','Thứ 7','CN']
day_colors = ['#3498db']*5 + ['#e67e22','#e67e22']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar AQI trung bình theo thứ
day_avg = df.groupby('day_name')['aqi'].mean().reindex(DAY_ORDER)
bars = axes[0].bar(DAY_ORDER, day_avg.values, color=day_colors, edgecolor='white')
axes[0].set_title('AQI trung bình theo thứ trong tuần', fontsize=12, fontweight='bold')
axes[0].set_ylabel('AQI trung bình', fontsize=11)
for bar, v in zip(bars, day_avg.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.8,
                 f'{v:.0f}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_ylim(0, day_avg.max()*1.15)
from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(fc='#3498db',label='Ngày thường'),
                         Patch(fc='#e67e22',label='Cuối tuần')], fontsize=9)

# Boxplot weekend vs weekday
df['week_type'] = df['is_weekend'].map({0:'Ngày thường (T2–T6)', 1:'Cuối tuần (T7–CN)'})
sns.boxplot(data=df, x='week_type', y='aqi',
            palette={'Ngày thường (T2–T6)':'#3498db','Cuối tuần (T7–CN)':'#e67e22'},
            width=0.4, linewidth=1.5, ax=axes[1])
axes[1].set_title('Phân phối AQI: Ngày thường vs Cuối tuần', fontsize=12, fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('AQI', fontsize=11)

wk_mean = df.groupby('week_type')['aqi'].mean()
for i, (label, val) in enumerate(wk_mean.items()):
    axes[1].text(i, axes[1].get_ylim()[0]+3, f'avg={val:.1f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

wk = df.groupby('is_weekend')['aqi'].mean()
print(f"→ AQI trung bình ngày thường: {wk[0]:.1f}")
print(f"→ AQI trung bình cuối tuần : {wk[1]:.1f}")
print(f"→ Chênh lệch: {wk[0]-wk[1]:+.1f} ({'cuối tuần sạch hơn' if wk[0]>wk[1] else 'cuối tuần ô nhiễm hơn'})")

## 9. EDA 8 — Heatmap AQI trung bình theo Giờ × Mùa
Giờ nào trong ngày ô nhiễm nhất, và mùa nào khuếch đại hiệu ứng đó?

In [ ]:
pivot = df.pivot_table(values='aqi', index='hour', columns='season_name',
                       aggfunc='mean')[SEASON_ORDER]

fig, ax = plt.subplots(figsize=(9, 10))
sns.heatmap(pivot, cmap='YlOrRd', annot=True, fmt='.0f',
            linewidths=0.3, annot_kws={'size': 8},
            cbar_kws={'label': 'AQI trung bình', 'shrink': 0.8}, ax=ax)
ax.set_title('Heatmap AQI trung bình theo Giờ × Mùa', fontsize=13, fontweight='bold')
ax.set_xlabel('Mùa', fontsize=12)
ax.set_ylabel('Giờ trong ngày', fontsize=12)
ax.set_yticklabels([f'{h}h' for h in range(24)], rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

# Tìm ô tệ nhất
max_val = pivot.max().max()
for col in SEASON_ORDER:
    for hour in range(24):
        if pivot.loc[hour, col] == max_val:
            print(f"→ Tệ nhất: {col} lúc {hour}h (AQI={max_val:.1f})")

## 10. Nhận xét phân tích tổng hợp

In [ ]:
hourly     = df.groupby('hour')['aqi'].mean()
peak_h     = int(hourly.idxmax())
best_h     = int(hourly.idxmin())
season_avg = df.groupby('season_name')['aqi'].mean()
worst_s    = season_avg.idxmax()
yearly_avg = df.groupby('year')['aqi'].mean()
x_num      = np.arange(len(monthly := df.groupby('ym')['aqi'].mean().reset_index()))
c_poly     = polyfit(x_num, monthly['aqi'].values, 1)
direction  = '↑ TĂNG' if c_poly[1] > 0 else '↓ GIẢM'
rush_avg   = df.groupby('is_rush_hour')['aqi'].mean()
wk_avg     = df.groupby('is_weekend')['aqi'].mean()

print("=" * 65)
print("  NHẬN XÉT PHÂN TÍCH EDA — HÀ NỘI AQI 2022–2025")
print("=" * 65)
print(f"  1. GIỜ Ô NHIỄM NHẤT  : {peak_h}h  (AQI={hourly[peak_h]:.1f})")
print(f"     Giờ sạch nhất      : {best_h}h  (AQI={hourly[best_h]:.1f})")
print()
print(f"  2. MÙA AQI CAO NHẤT  : {worst_s}")
print("     AQI trung bình theo mùa:")
for s in SEASON_ORDER:
    flag = " ← CAO NHẤT" if s == worst_s else ""
    print(f"       {s}: {season_avg[s]:.1f}{flag}")
print()
print(f"  3. XU HƯỚNG 4 NĂM    : {direction}  (slope={c_poly[1]:.4f}/tháng)")
for yr, v in yearly_avg.items():
    print(f"       {yr}: {v:.1f}")
print()
print(f"  4. GIỜ CAO ĐIỂM      : rush={rush_avg[1]:.1f}  vs  thường={rush_avg[0]:.1f}")
delta_rush = rush_avg[1]-rush_avg[0]
print(f"       Chênh lệch: {delta_rush:+.1f} ({'cao điểm ô nhiễm hơn' if delta_rush>0 else 'không ảnh hưởng rõ'})")
print()
delta_wk = wk_avg[0]-wk_avg[1]
print(f"  5. CUỐI TUẦN         : weekday={wk_avg[0]:.1f}  weekend={wk_avg[1]:.1f}")
print(f"       Chênh lệch: {delta_wk:+.1f} ({'ngày thường ô nhiễm hơn' if delta_wk>0 else 'cuối tuần ô nhiễm hơn'})")
print("=" * 65)